# Data Generation

This notebook is the executable entry point for raw SLIDE simulations. It generates the raw products consumed by `data_processing.ipynb` and writes them into `raw_data/` with parameter-descriptive filenames. Reusable simulation kernels live in `slide.data_generation`.

## Setup

Import reusable kernels, create output directories, and show where generated files will be written.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import jax.random as jr
import numpy as np
from tqdm.auto import tqdm

from slide.data_generation import (
    EMPIRICAL_NAMES,
    GENERATION_STEPS,
    RAW_FILENAMES,
    all_start_locs,
    expected_raw_outputs,
    generate_empirical_decay_curves,
    generate_empirical_strategy_sweep,
    generate_nk_decay_curves,
    generate_nk_strategy_sweep,
    load_empirical_landscape,
    missing_raw_outputs,
    nk_grid_pairs,
    run_nk_diffusion_replicates,
    uniform_start_locs,
)
from slide.utils import get_figures_dir, get_processed_data_dir, get_raw_data_dir, save_raw

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()

print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")

## Expected Raw Products

These are the raw products loaded by `data_processing.ipynb` and required for the paper figures.

In [ ]:
print(f"Expected raw products: {len(RAW_FILENAMES)}")
for key in sorted(RAW_FILENAMES):
    print(f"{key}: {RAW_FILENAMES[key]}")

## NK Grid Decay - Figure 3

Generate the large NK no-selection diffusion grid used for ruggedness accuracy, example decay curves, and NK summaries in Figure 3.

In [ ]:
rep_keys = jr.split(jr.PRNGKey(42), 25)
nk_decay_grid = []

for n_sites, k in tqdm(nk_grid_pairs(), desc="NK decay grid"):
    pair_results = []
    for key in rep_keys:
        run = run_nk_diffusion_replicates(
            key,
            n_sites=n_sites,
            k=k,
            num_alleles=2,
            start=np.zeros(n_sites, dtype=np.int32),
            popsize=2500,
            mutation_rate=0.5 / n_sites,
            num_reps=10,
            num_steps=25,
        )
        pair_results.append(run["fitness"].mean(axis=-1))
    nk_decay_grid.append(np.asarray(pair_results))

save_raw(np.asarray(nk_decay_grid), RAW_FILENAMES["nk_decay_grid"])

## NK Strategy Grid - Figure 5A

Generate the NK strategy lookup grid used to connect fitted ruggedness to directed-evolution control parameters.

In [ ]:
nk_strategy_grid = []

for n_sites, k in tqdm(nk_grid_pairs(), desc="NK strategy grid"):
    pair = generate_nk_strategy_sweep(
        n_sites=n_sites,
        num_alleles=2,
        k_values=[k],
        mutation_rate=0.1,
        popsize=1200,
        num_landscapes=1,
        num_reps=25,
        num_steps=25,
        strategy_grid_size=7,
        outer_reps=1,
        seed=42,
    )
    nk_strategy_grid.append(pair.reshape(7, 7, 25))

save_raw(np.asarray(nk_strategy_grid), RAW_FILENAMES["nk_strategy_grid"])

## NK Accuracy Sweeps - Figure 3C-D

Generate population-size and mutation-rate sensitivity sweeps for the fitted ruggedness estimate.

In [ ]:
rep_keys = jr.split(jr.PRNGKey(42), 25)

popsize_accuracy = []
for popsize in tqdm(np.linspace(100, 2500, 25, dtype=int), desc="NK popsize accuracy"):
    pop_results = []
    for key in rep_keys:
        run = run_nk_diffusion_replicates(
            key,
            n_sites=25,
            k=15,
            num_alleles=2,
            start=np.zeros(25, dtype=np.int32),
            popsize=int(popsize),
            mutation_rate=0.5 / 25,
            num_reps=20,
            num_steps=25,
        )
        pop_results.append(run["fitness"].mean(axis=-1))
    popsize_accuracy.append(np.asarray(pop_results))

save_raw(np.asarray(popsize_accuracy), RAW_FILENAMES["nk_popsize_accuracy"])

mutation_accuracy = []
for mu in tqdm(np.linspace(0.01, 2, 25), desc="NK mutation-rate accuracy"):
    mu_results = []
    for key in rep_keys:
        run = run_nk_diffusion_replicates(
            key,
            n_sites=25,
            k=15,
            num_alleles=2,
            start=np.zeros(25, dtype=np.int32),
            popsize=2000,
            mutation_rate=float(mu) / 25,
            num_reps=20,
            num_steps=25,
        )
        mu_results.append(run["fitness"].mean(axis=-1))
    mutation_accuracy.append(np.asarray(mu_results))

save_raw(np.asarray(mutation_accuracy), RAW_FILENAMES["nk_mutation_accuracy"])

## N4A20 Decay And Strategy Sweeps - Figure 5

Generate the N=4, A=20 NK decay and strategy products used for optimal-strategy summaries and the generation-count comparison.

In [ ]:
nk_decay_N4_A20 = generate_nk_decay_curves(
    n_sites=4,
    num_alleles=20,
    k_values=[1, 2, 3],
    mutation_rate=0.1,
    popsize=1200,
    num_starts=10000,
    num_reps=10,
    num_steps=25,
    seed=42,
)
save_raw(nk_decay_N4_A20, RAW_FILENAMES["nk_decay_N4_A20"])

nk_strategy_N4_A20 = generate_nk_strategy_sweep(
    n_sites=4,
    num_alleles=20,
    k_values=[1, 2, 3],
    mutation_rate=0.1,
    popsize=1200,
    num_landscapes=125,
    num_reps=10,
    num_steps=25,
    strategy_grid_size=7,
    outer_reps=10,
    seed=42,
)
save_raw(nk_strategy_N4_A20, RAW_FILENAMES["nk_strategy_N4_A20"])

for steps in tqdm(GENERATION_STEPS, desc="N4A20 generation-count strategy sweeps"):
    sweep = generate_nk_strategy_sweep(
        n_sites=4,
        num_alleles=20,
        k_values=[1, 2, 3],
        mutation_rate=0.1,
        popsize=1200,
        num_landscapes=10,
        num_reps=10,
        num_steps=int(steps),
        strategy_grid_size=7,
        outer_reps=10,
        seed=42,
    )
    save_raw(sweep, RAW_FILENAMES[f"nk_strategy_N4_A20_steps{steps}"])

## Empirical Decay Curves - Figure 4

Generate uniform-start and all-start no-selection diffusion curves for GB1, TrpB, TEV, and ParD3. These products feed empirical ruggedness, heterogeneity, and strategy-selection processing.

In [ ]:
empirical_landscapes = {name: load_empirical_landscape(name) for name in EMPIRICAL_NAMES}

for name, landscape in empirical_landscapes.items():
    popsize = 60 if name == "ParD3" else 2500
    starts_count = 8000 if name == "ParD3" else 10000

    uniform_starts = uniform_start_locs(landscape, num_starts=starts_count, seed=42)
    uniform_decay = generate_empirical_decay_curves(
        landscape,
        mutation_rate=0.1 / landscape.ndim,
        popsize=popsize,
        starts=uniform_starts,
        num_reps=10,
        num_steps=25,
        seed=42,
    )
    save_raw(uniform_decay, RAW_FILENAMES[f"empirical_decay_{name}_uniform"])

    all_starts = all_start_locs(landscape)
    all_decay = generate_empirical_decay_curves(
        landscape,
        mutation_rate=0.1 / landscape.ndim,
        popsize=popsize,
        starts=all_starts,
        num_reps=10,
        num_steps=25,
        seed=42,
    )
    save_raw(all_decay, RAW_FILENAMES[f"empirical_decay_{name}_all"])

## Empirical Strategy Sweeps - Figure 5D-G

Generate strategy sweeps from 100 uniformly sampled starts on each empirical landscape.

In [ ]:
for name, landscape in empirical_landscapes.items():
    starts = uniform_start_locs(landscape, num_starts=100, seed=42)
    strategy_sweep = generate_empirical_strategy_sweep(
        landscape,
        starts,
        mutation_rate=0.025,
        popsize=1200,
        num_reps=10,
        num_steps=25,
        strategy_grid_size=7,
        outer_reps=10,
        seed=42,
    )
    save_raw(strategy_sweep, RAW_FILENAMES[f"empirical_strategy_{name}_uniform"])

## Empirical Popsize Decay - Figure 4E

Generate empirical population-size sweeps for the fitted squared-decay ruggedness estimates.

In [ ]:
for name, landscape in empirical_landscapes.items():
    starts_count = 8000 if name == "ParD3" else 10000
    starts = uniform_start_locs(landscape, num_starts=starts_count, seed=42)
    popsize_results = []

    for popsize in tqdm(np.linspace(25, 2500, 10, dtype=int), desc=f"{name} popsize decay"):
        effective_popsize = max(1, int(popsize / 20)) if name == "ParD3" else int(popsize)
        popsize_results.append(
            generate_empirical_decay_curves(
                landscape,
                mutation_rate=0.1 / landscape.ndim,
                popsize=effective_popsize,
                starts=starts,
                num_reps=10,
                num_steps=25,
                seed=42,
            )
        )

    save_raw(np.asarray(popsize_results), RAW_FILENAMES[f"empirical_decay_{name}_popsize"])

## NK Heterogeneity - Figure 4

Generate the N=4, A=20 NK heterogeneity comparison product used alongside empirical heterogeneity analyses.

In [ ]:
nk_heterogeneity = generate_nk_decay_curves(
    n_sites=4,
    num_alleles=20,
    k_values=[1, 2, 3, 4],
    mutation_rate=0.1,
    popsize=1200,
    num_starts=10000,
    num_reps=10,
    num_steps=25,
    seed=42,
)
save_raw(nk_heterogeneity, RAW_FILENAMES["nk_heterogeneity"])

## Missing Raw Products

After running the generation cells above, this reports any expected raw product that is still absent from `raw_data/`.

In [ ]:
missing = missing_raw_outputs()
if missing:
    print(f"Missing raw products: {len(missing)}")
    for key, path in missing.items():
        print(f"{key}: {path}")
else:
    print("All expected raw products are present.")

print(f"Expected raw products: {len(expected_raw_outputs())}")